# XD2-Net — 03: Evaluation

Loads the trained models, evaluates them on the test set, and compares them against a
Logistic Regression baseline trained on the same aggregated (exp) features.

**Prerequisite:** run notebooks 01 and 02 first with the same `MODEL` / `DATASET_CONFIG`.

In [ ]:
import os, sys, glob, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab: clone the repo into /content (code) and write outputs to Drive (they survive the runtime).
REPO_URL = 'https://github.com/bemali/xd2-net-pytorch.git'
BRANCH = 'feattransformer-self-attention-exp'       # switch to 'main' once merged
OUTPUT_ROOT = None                                   # None = configs' workspace_dir (./outputs)

if IN_COLAB:
    REPO_ROOT = '/content/xd2-net-pytorch'
    if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
        !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
    %pip install -e {REPO_ROOT} --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/xd2net_outputs'
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
else:
    REPO_ROOT = os.path.abspath('..')

sys.path.insert(0, REPO_ROOT)   # importable without a kernel restart after pip install -e
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT, '| commit:',
      subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip())
print('OUTPUT_ROOT:', OUTPUT_ROOT)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU runtime)')
except ImportError:
    pass

In [ ]:
# --- parameters --------------------------------------------------------------
MODEL = 'baseline'                                   # 'baseline' | 'hierarchical'
DATASET_CONFIG = 'configs/datasets/bpic2012.yaml'    # any dataset YAML (see configs/datasets/_template.yaml)
MODEL_CONFIG = None                                  # None = configs/models/<MODEL>.yaml

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression

from xd2net.common.config import load_dataset_config, load_experiment_config
from xd2net.common.data import make_test_loader
from xd2net.common.metrics import evaluate_predictions
from xd2net.common.device import accelerated, resolve_device
from xd2net.common.registry import get_spec

spec = get_spec(MODEL)
exp = load_experiment_config(MODEL_CONFIG or spec.default_config, spec.model_config_cls)
ds = load_dataset_config(DATASET_CONFIG)
if OUTPUT_ROOT:
    exp.workspace_dir = OUTPUT_ROOT
run_dir = os.path.join(exp.workspace_dir, MODEL, ds.name)
artifacts_dir = os.path.join(run_dir, 'artifacts')
ckpt_dir = os.path.join(run_dir, 'checkpoints')
device = resolve_device(exp.training.device)
data_device = device if (exp.training.data_on_device and accelerated(device)) else None   # GPU-resident batches

def available_buckets():
    """Bucket keys with a saved artifact, in prefix-length order."""
    keys = [os.path.basename(p)[len('bucket_'):-len('.pkl')]
            for p in glob.glob(os.path.join(artifacts_dir, 'bucket_*.pkl'))]
    return sorted(keys, key=lambda k: int(k.split('_')[1]))

In [ ]:
def load_model_for_bucket(bucket_key):
    """Rebuild the model from its saved config and load the best checkpoint."""
    with open(os.path.join(ckpt_dir, f'model_config_{bucket_key}.json')) as f:
        model = spec.model_cls.from_config(json.load(f))
    checkpoints = glob.glob(os.path.join(ckpt_dir, f'model_{bucket_key}_*.pt'))
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint for {bucket_key} in {ckpt_dir}')
    best = min(checkpoints, key=lambda p: float(p.split('vl')[-1].replace('.pt', '')))
    model.load_state_dict(torch.load(best, map_location='cpu', weights_only=True))
    print(f'  Loaded: {os.path.basename(best)}')
    return model

In [ ]:
xd2_results = []
lr_results   = []

for bucket_key in available_buckets():
    with open(os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl'), 'rb') as f:
        art = pickle.load(f)

    print(f'\nBucket {bucket_key}')

    # ── XD2-Net evaluation ────────────────────────────────────────────────────
    model = load_model_for_bucket(bucket_key)

    test_loader = make_test_loader(
        art['inputs_test'], art['outputs_test'], art['lengths_test'],
        batch_size=exp.training.batch_size, device=data_device,
    )
    explainer = spec.explainer_cls(model, device=exp.training.device)
    df_pred, _ = explainer.explain_loader(
        test_loader, art['caseids_test'], art['feature_names'], bucket=bucket_key
    )

    m = evaluate_predictions(df_pred['GROUND_TRUTH'].values, df_pred['PREDICTION_PROB'].values,
                             target_names=ds.outcome.labels)
    xd2_results.append({'bucket': bucket_key, 'accuracy': m['accuracy'],
                        'precision': m['precision'], 'recall': m['recall'],
                        'f1': m['f1'], 'roc_auc': m['roc_auc']})
    print(f'  XD2-Net  acc={m["accuracy"]:.3f}  auc={m["roc_auc"]:.3f}  f1={m["f1"]:.3f}')

    # ── Logistic Regression baseline (on exp features) ────────────────────────
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(art['inputs_train']['input_exp'], art['outputs_train']['act_output'])
    lr_prob = lr.predict_proba(art['inputs_test']['input_exp'])[:, 1]
    ml = evaluate_predictions(art['outputs_test']['act_output'], lr_prob,
                              target_names=ds.outcome.labels)
    lr_results.append({'bucket': bucket_key, 'accuracy': ml['accuracy'],
                       'precision': ml['precision'], 'recall': ml['recall'],
                       'f1': ml['f1'], 'roc_auc': ml['roc_auc']})
    print(f'  LR       acc={ml["accuracy"]:.3f}  auc={ml["roc_auc"]:.3f}  f1={ml["f1"]:.3f}')

In [ ]:
# ── Side-by-side comparison table ────────────────────────────────────────────
df_xd2 = pd.DataFrame(xd2_results).set_index('bucket').add_prefix('xd2_')
df_lr  = pd.DataFrame(lr_results).set_index('bucket').add_prefix('lr_')
df_cmp = df_xd2.join(df_lr)
print(df_cmp.round(3).to_string())

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
out_dir = os.path.join(run_dir, 'results')
os.makedirs(out_dir, exist_ok=True)
df_cmp.to_csv(os.path.join(out_dir, 'comparison_results.csv'))
print('Saved to', out_dir)